In [1]:
# ===== IMPORTS =====
import os
import json
import cv2
import torch
import numpy as np
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===== PATH =====
BASE_PATH = "/kaggle/input/datasets/kunalnarang47/deepfashion-redwing/deepfashion2_pruned"

# ===== CATEGORY MAP =====
TOP5 = [1, 8, 7, 2, 9]
cat_to_idx = {cat: i for i, cat in enumerate(TOP5)}

# ===== MULTI-LABEL =====
def create_multilabel(data, cat_to_idx):
    label = np.zeros(len(cat_to_idx))
    for key in data:
        if "item" in key:
            cat = data[key]["category_id"]
            if cat in cat_to_idx:
                label[cat_to_idx[cat]] = 1
    return label

# ===== DATASET =====
class FashionDataset(Dataset):
    def __init__(self, base_path, split, cat_to_idx):
        self.ann_path = os.path.join(base_path, split, "annos")
        self.img_path = os.path.join(base_path, split, "image")
        self.files = os.listdir(self.ann_path)
        self.cat_to_idx = cat_to_idx

    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        file = self.files[idx]
        
        with open(os.path.join(self.ann_path, file)) as f:
            data = json.load(f)
        
        label = create_multilabel(data, self.cat_to_idx)
        
        img_file = file.replace(".json", ".jpg")
        img = cv2.imread(os.path.join(self.img_path, img_file))

        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)

        img = cv2.resize(img, (224, 224))

        # normalization
        img = img / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std

        img = torch.tensor(img).permute(2,0,1).float()
        label = torch.tensor(label).float()
        
        return img, label

# ===== LOADER =====
dataset = FashionDataset(BASE_PATH, "train", cat_to_idx)

train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# ===== LOSS =====
weights = torch.tensor([
    71645/71645,
    71645/55387,
    71645/36616,
    71645/36064,
    71645/30835
]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=weights)

Using device: cuda


In [2]:
print("Starting EfficientNet Transfer...")

model = models.efficientnet_b0(weights='DEFAULT')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10

for epoch in range(num_epochs):
    print(f"[Transfer] Epoch {epoch+1}")
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        if i % 500 == 0:
            print(f"Batch {i}")

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Loss:", total_loss/len(train_loader))

torch.save(model.state_dict(), "/kaggle/working/efficientnet_transfer_10.pth")
print("✅ EfficientNet Transfer Saved")

Starting EfficientNet Transfer...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 147MB/s]


[Transfer] Epoch 1
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.3831823369426728
[Transfer] Epoch 2
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.26454636111199004
[Transfer] Epoch 3
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.21269549393945575
[Transfer] Epoch 4
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.17685786802468434
[

In [3]:
print("Starting EfficientNet Scratch...")

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

for epoch in range(num_epochs):
    print(f"[Scratch] Epoch {epoch+1}")
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        if i % 500 == 0:
            print(f"Batch {i}")

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Loss:", total_loss/len(train_loader))

torch.save(model.state_dict(), "/kaggle/working/efficientnet_scratch_10.pth")
print("✅ EfficientNet Scratch Saved")

Starting EfficientNet Scratch...
[Scratch] Epoch 1
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.7039626195353761
[Scratch] Epoch 2
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.5757913086929222
[Scratch] Epoch 3
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000
Loss: 0.49338607624670916
[Scratch] Epoch 4
Batch 0
Batch 500
Batch 1000
Batch 1500
Batch 2000
Batch 2500
Batch 3000
Batch 3500
Batch 4000
Batch 4500
Batch 5000
Batch 5500
Batch 6000
Batch 6500
Batch 7000
Batch 7500
Batch 8000
Batch 8500
Batch 9000